# model2.0.ipynb — UG2 Flotation Models (Modular)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score
import torch

from src.models.tree_models import get_xgboost_model
from src.models.linear_models import get_ridge_model
from src.models.lstm import run_lstm_loocv
from src.utils.visualization import plot_actual_vs_predicted

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Ready on {DEVICE}')

In [ ]:
EXCEL_PATH = '../data/UG2_ML_Structured_Dataset.xlsx'
streams = pd.read_excel(EXCEL_PATH, sheet_name='Stream-Level Dataset', header=2)
assays  = pd.read_excel(EXCEL_PATH, sheet_name='Assay Database',       header=2)
streams.columns = [c.replace('\n', ' ').strip() for c in streams.columns]
assays.columns  = [c.replace('\n', ' ').strip() for c in assays.columns]
print(f'Streams: {streams.shape} | Assays: {assays.shape}')

## 1. Stream-Level Model (n=33)

In [ ]:
conc = streams[streams['Stream_Type'].str.contains('Conc', na=False)].copy()
bank_map = {'Rougher 1 Conc': 0, 'Rougher 2 Conc': 1, 'Flash Float Conc': 2}
cell_map = {'Cell 1': 1, 'Cell 2': 2, 'Cell 3': 3, 'Cell 4': 4, 'Cell 5': 5, 'FF Cell 1': 1}
conc['Bank_Code'] = conc['Stream_Type'].map(bank_map).fillna(0).astype(int)
conc['Cell_Num'] = conc['Bank_Cell'].map(cell_map).fillna(3).astype(int)

X_str = conc[['Run', 'Bank_Code', 'Cell_Num', 'Dry_Mass (t/h)', 'Flow_Rate (m³/h)', 'Pulp_Density (t/m³)', 'Cr2O3 (%)']].values.astype(float)
y_str = conc['PGM+Au (g/t)'].values.astype(float)

scaler_str = StandardScaler()
X_str_scaled = scaler_str.fit_transform(X_str)
loo = LeaveOneOut()

In [ ]:
str_ridge = get_ridge_model(alpha=0.5)
y_str_pred_ridge = cross_val_predict(str_ridge, X_str_scaled, y_str, cv=loo)

str_xgb = get_xgboost_model(n_estimators=30, max_depth=2, learning_rate=0.1, reg_lambda=1.0)
y_str_pred_xgb = cross_val_predict(str_xgb, X_str_scaled, y_str, cv=loo)

print('Training LSTM LOOCV for Stream data...')
y_str_pred_lstm = run_lstm_loocv(X_str_scaled, y_str, epochs=80, device=DEVICE)

print(f"Ridge   R²: {r2_score(y_str, y_str_pred_ridge):.3f}")
print(f"XGBoost R²: {r2_score(y_str, y_str_pred_xgb):.3f}")
print(f"LSTM    R²: {r2_score(y_str, y_str_pred_lstm):.3f}")

## 2. Assay-Level Model (n=32)

In [ ]:
needed = ['Cu (%)', 'Ni (%)', 'Fe (%)', 'Cr2O3 (%)', 'PGM+Au (g/t)']
asy_clean = assays.dropna(subset=needed).copy()
asy_clean['S_over_Cr2O3'] = asy_clean['S (%)'].fillna(0) / (asy_clean['Cr2O3 (%)'] + 1e-6)
asy_clean['Cu_over_Ni'] = asy_clean['Cu (%)'] / (asy_clean['Ni (%)'] + 1e-6)

X_asy = asy_clean[['Cu (%)', 'Ni (%)', 'Fe (%)', 'Cr2O3 (%)', 'S_over_Cr2O3', 'Cu_over_Ni']].values.astype(float)
y_asy = asy_clean['PGM+Au (g/t)'].values.astype(float)

scaler_asy = StandardScaler()
X_asy_scaled = scaler_asy.fit_transform(X_asy)

In [ ]:
asy_ridge = get_ridge_model(alpha=1.0)
y_asy_pred_ridge = cross_val_predict(asy_ridge, X_asy_scaled, y_asy, cv=loo)

asy_xgb = get_xgboost_model(n_estimators=30, max_depth=2, learning_rate=0.1, reg_lambda=1.0)
y_asy_pred_xgb = cross_val_predict(asy_xgb, X_asy_scaled, y_asy, cv=loo)

print('Training LSTM LOOCV for Assay data...')
y_asy_pred_lstm = run_lstm_loocv(X_asy_scaled, y_asy, epochs=80, device=DEVICE)

print(f"Ridge   R²: {r2_score(y_asy, y_asy_pred_ridge):.3f}")
print(f"XGBoost R²: {r2_score(y_asy, y_asy_pred_xgb):.3f}")
print(f"LSTM    R²: {r2_score(y_asy, y_asy_pred_lstm):.3f}")